# HKI Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/h3nok/HKI/blob/main/notebooks/01_quickstart.ipynb)

**Hermetic Knowledge Isolation (HKI)** is a runtime contract that ensures every AI agent request runs inside exactly one domain — and can only read, retrieve, cache, or call tools scoped to that domain.

This notebook shows the problem, the fix, and the four primitives you need to enforce it.

**Time to complete:** ~5 minutes  
**Requirements:** Python 3.11+

In [ ]:
# Install
%pip install hki-runtime -q

---
## Part 1 — The problem: your AI agents share more than you think

Consider a multi-tenant AI platform with a **pharmacy** domain and a **travel** domain.
Without isolation, a simple cache bug lets one domain's answer bleed into another.

In [ ]:
# WITHOUT HKI: cache key is only the query string — domain is not part of the key

CACHE = {}

def retrieve_buggy(query: str, domain: str) -> str:
    """Simulates a RAG retrieval call — the cache key is wrong."""
    if query in CACHE:           # BUG: only the query, not the domain
        return CACHE[query]
    answer = f"[{domain} answer] {query}"
    CACHE[query] = answer
    return answer

pharmacy_answer = retrieve_buggy("what is the return policy", domain="pharmacy")
travel_answer   = retrieve_buggy("what is the return policy", domain="travel")

print(f"pharmacy → {pharmacy_answer}")
print(f"travel   → {travel_answer}")
print()
print(f"Same answer? {pharmacy_answer == travel_answer}  ← travel got pharmacy's cached response")

The travel agent just received a pharmacy-scoped answer from the cache. In production this means:
- Drug pricing visible to travel agents
- Member health data surfaced across business units  
- Compliance violations with no audit trail

This is **HKI Threat T01** — semantic cache cross-domain leak. There are 14 more like it.

---
## Part 2 — The fix: a signed scope envelope on every request

In [ ]:
from hki_runtime import validate_envelope, derive_hki_cache_key

# An HKI envelope travels with every request — signed at the gateway,
# validated at every downstream service.

def make_envelope(domain: str) -> dict:
    return {
        "hki_version": "1.0",
        "envelope_id": f"env_{domain}_001",
        "org_id": "org_acme",
        "subject_id": "user_42",
        "active_domain": domain,
        "authorized_domains": [domain],
        "purpose": "retrieve",
        "risk_tier": "read-only",
        "policy_pack_id": f"{domain}@2026-05",
        "issued_at": 0,
        "expires_at": 9_999_999_999,
        "issuer": "gateway.acme.internal",
        "signature": "ed25519:placeholder",
    }

# Validate the envelope — fails closed on missing/expired/global scope
result = validate_envelope(make_envelope("pharmacy"))
print(f"Valid: {result.ok}")
print(f"Active domain: {result.envelope.active_domain}")

In [ ]:
# WITH HKI: cache key includes org + domain — cross-domain collisions are impossible

SAFE_CACHE = {}

def retrieve_safe(query: str, domain: str) -> str:
    envelope = make_envelope(domain)
    key = derive_hki_cache_key({
        "envelope": envelope,
        "operation": "retrieval.search",
        "input": {"query": query},
    })
    if key in SAFE_CACHE:
        return SAFE_CACHE[key]
    answer = f"[{domain} answer] {query}"
    SAFE_CACHE[key] = answer
    return answer

pharmacy_answer = retrieve_safe("what is the return policy", domain="pharmacy")
travel_answer   = retrieve_safe("what is the return policy", domain="travel")

print(f"pharmacy → {pharmacy_answer}")
print(f"travel   → {travel_answer}")
print()
print(f"Same answer? {pharmacy_answer == travel_answer}  ← isolated")

---
## Part 3 — Artifact visibility

Every document, embedding, or memory entry must be labelled with a domain.
HKI enforces **exact-equality reads** — being in `authorized_domains` is not enough.

In [ ]:
from hki_runtime import assert_artifact_visible

pharmacy_env = validate_envelope(make_envelope("pharmacy")).envelope

# Reading a pharmacy document while in the pharmacy domain — allowed
issue = assert_artifact_visible(pharmacy_env, {
    "org_id": "org_acme",
    "domain": "pharmacy",
    "artifact_type": "document",
    "artifact_id": "rx_policy_001",
})
print(f"pharmacy doc in pharmacy domain → {'allowed' if issue is None else issue.message}")

# Reading a travel document while in the pharmacy domain — BLOCKED
issue = assert_artifact_visible(pharmacy_env, {
    "org_id": "org_acme",
    "domain": "travel",          # different domain
    "artifact_type": "document",
    "artifact_id": "hotel_rates_q2",
})
print(f"travel doc in pharmacy domain   → {'allowed' if issue is None else 'blocked: ' + issue.message}")

---
## Part 4 — Gateway target enforcement

Every tool call, model route, and retriever target is checked against the active domain.
Scope arguments in the request body cannot override the signed envelope.

In [ ]:
from hki_runtime import evaluate_gateway_target, reject_conflicting_scope_argument

env = validate_envelope(make_envelope("pharmacy")).envelope

# Tool call in the right domain — allowed
decision = evaluate_gateway_target(env, {
    "type": "tool",
    "id": "rx.lookup",
    "domain": "pharmacy",
})
print(f"rx.lookup (pharmacy) → {decision.allowed}")

# Tool call in a different domain — BLOCKED
decision = evaluate_gateway_target(env, {
    "type": "tool",
    "id": "hotel.search",
    "domain": "travel",
})
print(f"hotel.search (travel in pharmacy context) → allowed={decision.allowed}, reason={decision.reason}")

# Scope override attempt — request body tries to claim a different domain
override = reject_conflicting_scope_argument(env, {"domain": "travel", "query": "..."}) 
print(f"\nScope override attempt → {override or 'no conflict'}")

---
## Part 5 — What HKI fails closed on

HKI rejects anything that could silently grant access across domains.

In [ ]:
from hki_runtime import validate_envelope

test_cases = [
    ("missing active_domain",   {**make_envelope("pharmacy"), "active_domain": None}),
    ("global domain",           {**make_envelope("pharmacy"), "active_domain": "global"}),
    ("wildcard domain",         {**make_envelope("pharmacy"), "active_domain": "*"}),
    ("domain not in authorized",{**make_envelope("pharmacy"), "active_domain": "travel"}),
    ("expired envelope",        {**make_envelope("pharmacy"), "expires_at": 1}),
    ("empty authorized_domains",{**make_envelope("pharmacy"), "authorized_domains": []}),
]

for label, bad_env in test_cases:
    result = validate_envelope(bad_env)
    status = "PASS (rejected)" if not result.ok else "FAIL (accepted — this is a bug)"
    issues = "; ".join(i.message for i in result.issues) if not result.ok else ""
    print(f"  {label:<35s} → {status}")
    if issues:
        print(f"    reason: {issues}")

---
## Summary

| What you did | HKI primitive used |
|---|---|
| Validated a signed scope envelope at the edge | `validate_envelope` |
| Derived a domain-bound cache key | `derive_hki_cache_key` |
| Checked artifact visibility by exact domain | `assert_artifact_visible` |
| Enforced tool call domain at the gateway | `evaluate_gateway_target` |
| Rejected a scope override from the request body | `reject_conflicting_scope_argument` |

These five functions are the entire public surface of `hki-runtime`. Every higher-level adapter (LangChain, LlamaIndex, LiteLLM, AutoGen, CrewAI, Google ADK) calls these same primitives.

### Next notebooks

- [02 — LangChain RAG integration](./02_langchain_rag.ipynb)
- [03 — FastAPI middleware](./03_fastapi_middleware.ipynb)
- [04 — Threat demos (all 15)](./04_threat_demos.ipynb)

### Links

- GitHub: https://github.com/h3nok/HKI
- Standard: https://github.com/h3nok/HKI/blob/main/spec/HKI-1.0.md
- Threat catalog: https://github.com/h3nok/HKI/blob/main/docs/HKI_THREATS.md